In [1]:
import pandas as pd
import random
import csv
from tqdm import tqdm
import string
import numpy as np

from collections import Counter

import spacy
# spacy.cli.download("nl_core_news_lg")
nlp = spacy.load("nl_core_news_lg") 

import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ['hf_token_own']
)

In [3]:
path_name = '/Users/sabijn/Documents/PhD/Datasets/chisor_dataset_all/ChiSCor_CoNLL_paper/csv/ChiSCor_master_df_password/ChiSCor_master_df.csv'
df = pd.read_csv(path_name, index_col=0)

In [4]:
example = "Er is een meisje dat op straat speelt."
for token in nlp(example):
    print(token, token.pos_)

Er ADV
is VERB
een DET
meisje NOUN
dat PRON
op ADP
straat NOUN
speelt VERB
. PUNCT


In [5]:
from collections import defaultdict
pos_tags = []
pos_tag_story_begin = []
stories_with_er = defaultdict(list)
for story in df['story_raw']:
    doc = nlp(story)

    sent_list = list(doc.sents)

    for i, sent in enumerate(sent_list):
        first_word = sent[0]
        if i == 0:
            if str(first_word).lower() == 'er':
                stories_with_er[first_word.pos_].append(story)

        pos_tags.append(first_word.pos_)
pos_counter = Counter(pos_tag_story_begin)

In [6]:
stories_with_er.keys()

dict_keys(['ADV'])

In [8]:
nouns = set()
adjectives = set()
verbs = set()

for story in df['story_raw']:
    for token in nlp(story):
        if token.pos_ == 'NOUN':
            nouns.add(str(token))
        elif token.pos_ == 'VERB':
            verbs.add(str(token))
        elif token.pos_ == 'ADJ':
            adjectives.add(str(token))

KeyboardInterrupt: 

In [ ]:
nouns, adjectives, verbs = list(nouns), list(adjectives), list(verbs)

In [ ]:
def select_pos_tag(weighted=True):
    if weighted:
        tags = list(pos_counter.keys())
        frequencies = list(pos_counter.values())
        sample = random.choices(tags, weights=frequencies, k=1)[0]
    else:
        sample = pos_tags[random.randint(0, len(pos_tags) - 1)]
    
    return sample

In [ ]:
# you took these from Finke (2025), translated them, and removed a few. 
narrative_elements = [
    "in medias res",
    "een morele les",
    "een onverwachte wending",
    "een onbetrouwbare verteller",
    "vooruitwijzing",
    "innerlijke monoloog",
    "symboliek",
    "een niet-lineaire tijdlijn",
    "een omgekeerde tijdlijn",
    "circulaire verhaalsstructuur",
    "een flashback",
    "een geneste structuur",
    "meerdere perspectieven",
    "een cliffhanger",
    "contrast (juxtapositie)",
    "climax-structuur"
]

verhaalthemas = [
    "sprekende dieren",
    "fantasiewerelden",
    "tijdreizen",
    "een deadline of tijdslimiet",
    "ruimteverkenning",
    "mystieke wezens",
    "onderwateravonturen",
    "dinosaurussen",
    "piraten",
    "superhelden",
    "sprookjes",
    "het heelal",
    "verborgen schatten",
    "magische landen",
    "betoverde bossen",
    "geheime genootschappen",
    "robots en technologie",
    "sport",
    "schoolleven",
    "vakanties",
    "culturele tradities",
    "magische voorwerpen",
    "verloren beschavingen",
    "ondergrondse werelden",
    "vervlogen tijdperken",
    "onzichtbaarheid",
    "reusachtige wezens",
    "miniatuurwerelden",
    "ontmoetingen met buitenaardse wezens",
    "behekste plekken",
    "vormverandering",
    "eilandavonturen",
    "ongewone voertuigen",
    "geheime missies",
    "droomwerelden",
    "virtuele werelden",
    "raadsels",
    "rivaliteit tussen broers en zussen",
    "schattenjachten",
    "sneeuwavonturen",
    "seizoenswisselingen",
    "mysterieuze kaarten",
    "koninkrijken",
    "levende objecten",
    "tuinen",
    "verloren steden",
    "de kunsten",
    "de hemel"
]

In [9]:
# def generate_user_prompt():
#     chosen_noun = random.choice(nouns)
#     chosen_adjective = random.choice(adjectives)
#     chosen_verb = random.choice(verbs)
#     chosen_pos_tag = select_pos_tag()
#     chosen_letter = random.choice(string.ascii_lowercase)
#     chosen_feature = random.choice(story_features)
#     element = random.choice(verhaalelementen)

#     prompt = f"""Vertel een verhaal. 
# Het verhaal moet het volgende werkwoord bevatten: {chosen_verb}, het volgende zelfstandig naamwoord: {chosen_noun} en het volgende bijvoegelijk naamwoord: {chosen_adjective}.
# Het verhaal moet het volgende kenmerk bevatten: {chosen_feature} en het volgende verhaal element: {element}.
# Begin het verhaal met een woord met het volgende pos-tag {chosen_pos_tag}."""

#     return prompt

# def generate_tweeked_user_prompt(*, narrative_elements=None):
#     random.seed(10)
#     element = random.choice(narrative_elements)
#     prompt = f"""Vertel een verhaal. Het verhaal moet het volgende narratieve element bevatten: {element}"""

#     return prompt

def generate_tweeked_user_prompt(chosen_pos_tag):
    prompt = f"""Vertel een verhaal. 
Begin het verhaal met een woord met het volgende pos-tag {chosen_pos_tag}."""

    return prompt

def generate_user_prompt():
    prompt = f"""Vertel een verhaal."""

    return prompt

In [ ]:
experiment = "llama"

if experiment == "llama":
    name = "llama3-8b"
    model="meta-llama/Llama-3.1-8B-Instruct:novita"
    # name="llama3-70b"
    # model="meta-llama/Llama-3.3-70B-Instruct:novita"
elif experiment == "gemma":
    name = "gemma3-27b"
    model="google/gemma-3-27b-it:nebius"

print(name)
#user_prompt = generate_user_prompt()

system_prompt = f"""
Je bent een verteller van een kort verhaal (rond de 200 woorden).
Je bent een kind tussen de 4 en 6 en je vertelt een verhaal aan klasgenoten. 
Je publiek bestaat uit kinderen van jouw leeftijd. 
Geef het verhaal geen titel of introductie.
"""

with open(f"/Users/sabijn/Documents/PhD/code/storylm_p1_data/prompting/prompting_results/baseline_research_prompting_{name}.csv", "a", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    
    # Write header only once if file is empty
    csvfile.seek(0, 2)  # move to end
    if csvfile.tell() == 0:  
        writer.writerow(["model", "system", "user", "completion1", "completion2", "completion3", "completion4", "completion5"])

    for pos in tqdm(pos_counter.keys()):
        completions = []
        user_prompt = generate_tweeked_user_prompt(pos)

        for _ in tqdm(range(5)):
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
            )

            completion = response.choices[0].message.content.strip()
            completions.append(completion)

        # Write each row immediately
        writer.writerow([response.model, system_prompt, user_prompt] + completions)


llama3-8b


NameError: name 'pos_counter' is not defined

In [62]:

name = "llama3-8b"
model="meta-llama/Llama-3.1-8B-Instruct:novita"

system_prompt = f"""
Je bent een verteller van een kort verhaal (rond de 200 woorden).
Je bent een kind tussen de 4 en 6 en je vertelt een verhaal aan klasgenoten. 
Je publiek bestaat uit kinderen van jouw leeftijd. 
Geef het verhaal geen titel of introductie.
"""

pos = f"""Vertel een verhaal. 
Begin het verhaal met een woord met het POS tag PREP."""
user_prompt = generate_tweeked_user_prompt(pos)

for _ in tqdm(range(5)):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.7,
    )

    completion = response.choices[0].message.content.strip()
    print('************* Completion 1', completion)
    cleaned = completion.strip('\n').lower().split(' ')
    if cleaned[0] == 'er':
        print('yes')
        print('verhaal:', completion)
        response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": f'Wat is de POS tag van het eerste woord van dit verhaal? Geef het woord en de bijbehorende POS tag. Verhaal: {completion}'},
        ],
        )
        print(response.choices[0].message.content.strip())


 20%|██        | 1/5 [00:04<00:19,  4.82s/it]

************* Completion 1 Vandaag ging ik naar de speeltuin met mijn vriendin Lente. We liepen door de grote draaimolen, die zo hard ging dat het geluid van de motor net zo hard was als mijn hartje. 

Lente gilde zo hard van plezier dat ik moest lachen. Ik kreeg een idee in mijn hoofd. "Lente, weet je wat? Laten we de draaimolen op de snelste stand zetten!" riep ik. Lente knikte en we stopten ons geld in de machine. De man achter de draaimolen knikte en ging langzaam de knoppen in de draaimolen aan.

"Wheeee!" gilde Lente, toen de draaimolen zo hard ging dat het leek alsof we de lucht in vlogen. Ik gilde ook van plezier. Daar stonden we, twee meisjes in een draaimolen, die zo hard ging dat het leek alsof we de zon in konden vliegen.


 40%|████      | 2/5 [00:09<00:13,  4.49s/it]

************* Completion 1 Mijn zusje heeft een prachtig paard. 
Het heet Liefde en het is zo zacht als een veer. 

Liefde woont op een grote wei waar hij altijd gras eet. 

Hij heeft vriendjes die ook paarden zijn en samen spelen ze. 

Maar Liefde is een beetje anders. Hij kan vliegen. 

Echt waar! Hij heeft een speciale kruisboog op zijn rug en hij kan als een vleermuis door de lucht vliegen. 

Een keer vloog Liefde zo hoog dat hij de wolken zag. 

Hij zag een vriendelijke kabouter die hem aanmoedigde verder te vliegen. 

Liefde vloog nog hoger en hij zag zelfs de sterren. 

Hij voelde zich alsof hij de hele wereld kon zien. 

Toen moest Liefde terug naar zijn wei. 

Hij had zin in een lekker grasje en een kop warm water. 

En dat is het verhaal over Liefde, het vliegende paard.


 60%|██████    | 3/5 [00:13<00:08,  4.39s/it]

************* Completion 1 Vertel een verhaal. 

Ons konijntje ging naar de speeltuin. Daar speelde hij met zijn beste vriend, een hondje. Ze speelden met bal en kruisbalk.

Toen hoorde ons konijntje een stemmen die zei: "Wie wil een oliebollen maken?" Het konijntje en de hond keken elkaar aan en zei: "Wij willen een oliebollen maken!" 

Ze liepen naar de kookkar en begonnen met het mengen van oliebollen. Maar toen ze probeerden om de oliebollen in de olie te doen, bleven ze plakken aan de randen van de bak. Het konijntje en de hond keken elkaar aan en lachten. "Dit is een grote problemen!" zei het konijntje. 

Maar dan had de hond een idee. "We moeten onze handen eerst in de plakkersaus doen!" En daarmee lukte het ons konijntje en de hond om de oliebollen eindelijk in de olie te doen.


 80%|████████  | 4/5 [00:17<00:04,  4.24s/it]

************* Completion 1 Ergens in een groot bos, waar de bomen zo hoog waren dat ze bijna de wolken raakten, leefde een prinsesje. 

De prinsesje heette Sophia en ze was erg lief, maar ook erg stom. Want Sophia dacht dat een gans een vogel was die kon vliegen, maar ze had nog nooit een vogel gezien. 

Een dag ging Sophia in de bossen op zoek naar een vogel, maar in plaats daarvan kwam ze een grote gans tegen die heel erg boos op haar was. 

De gans was zo boos dat hij Sophia in zijn bek nam en met hem meeging naar zijn gansenvrienden. 

Sophia was erg bang, maar toen ze de gansenvrienden zag, begon ze te giechelen. Want zoveel gansenvrienden had ze nog nooit gezien. 

De gansenvrienden vonden het ook erg leuk dat Sophia lachte en ze besloten om alle gansenvrienden mee te nemen naar het kasteel van de prins.


100%|██████████| 5/5 [00:21<00:00,  4.22s/it]

************* Completion 1 Onder de grote boom in ons park woonden de konijnen. Het waren de liefste konijnen ter wereld. Ze hadden grote oren en mooie staarten. Hun naam was Fluffy en hij had een broertje, Snoopy.

Ze leefden in een prachtige grot onder de boom. Fluffy en Snoopy waren altijd aan het spelen. Ze renden rond, ze sprongen over stenen en ze gingen op avontuur. Ze wilden altijd iets nieuws ontdekken.

Een dag, toen ze uit waren, ontdekten ze een klein poeltje water. Daaromheen groeiden prachtige bloemen. Fluffy en Snoopy waren zo blij dat ze elkaar begonnen te knuffelen. Ze waren zo gelukkig dat ze nooit meer wilden stoppen met spelen.

En zo leefden Fluffy en Snoopy, de liefste konijnen ter wereld, verder in hun grot onder de grote boom.


### Age experiment

In [ ]:
stories = []
models = []
ages_by_stories = []

In [11]:
n = 10

user_prompt = generate_user_prompt()

for age_set in [[6, 7], [7, 8], [8, 9], [9, 10], [10, 11], [11, 12]]:
    lower_age = age_set[0]
    upper_age = age_set[1]
    system_prompt = f"""
Je bent een verteller van een kort verhaal (rond de 200 woorden).
Je bent een kind tussen de {lower_age} en {upper_age} en je vertelt een verhaal aan klasgenoten. 
Je publiek bestaat uit kinderen van jouw leeftijd. 
Geef het verhaal geen titel of introductie.
"""

    for _ in tqdm(range(n)):
        response = client.chat.completions.create(
            model="meta-llama/Llama-3.1-8B-Instruct:novita",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )

        completion = response.choices[0].message.content.strip()
        models.append(response.model)
        stories.append(completion)
        ages_by_stories.append(f'{lower_age}_{upper_age}')

100%|██████████| 10/10 [00:43<00:00,  4.33s/it]


In [12]:
ages_lama = pd.DataFrame(
    {'model': models, 
    'age': ages_by_stories,
    'story': stories}
)

In [13]:
ages_lama.to_csv('prompting_results/age_research_test_mistake.csv')

## Baseline (no told by kids)

In [6]:
experiment = "llama"

if experiment == "llama":
    name = "llama3-8b"
    model="meta-llama/Llama-3.1-8B-Instruct:novita"
    # name="llama3-70b"
    # model="meta-llama/Llama-3.3-70B-Instruct:novita"
elif experiment == "gemma":
    name = "gemma3-27b"
    model="google/gemma-3-27b-it:nebius"

system_prompt = f"""
Je bent een verteller van een kort verhaal (rond de 200 woorden).
Gebruik alleen hele simpele woorden die een 3-jarig kind kan begrijpen.
Geef het verhaal geen titel of introductie.
"""

user_prompt = generate_user_prompt()
print(user_prompt)

with open(f"/Users/sabijn/Documents/PhD/code/storylm_p1_data/prompting/prompting_results/baseline_research_prompting_{name}.csv", "a", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    
    # Write header only once if file is empty
    csvfile.seek(0, 2)  # move to end
    if csvfile.tell() == 0:  
        writer.writerow(["model", "system", "user", "completion"])

    for _ in tqdm(range(600)):
        completions = []
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )

        completion = response.choices[0].message.content.strip()
        completions.append(completion)

        # Write each row immediately
        writer.writerow([response.model, system_prompt, user_prompt] + completions)


Vertel een verhaal.


100%|██████████| 600/600 [39:27<00:00,  3.95s/it]
